<a href="https://colab.research.google.com/github/ryukf333/remote-sensing-crop-yield-ml/blob/main/01yield_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Project_2_Crop_Yield"

Mounted at /content/drive


In [ ]:
import pandas as pd
import os


RAW_DIR = f"{PROJECT_DIR}/data_raw"
PROCESSED_DIR = f"{PROJECT_DIR}/data_processed"

file_path = f"{RAW_DIR}/nass_iowa_corn_yield_2019_2024.csv"

df = pd.read_csv(file_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head(2))

Rows: 533
Columns: 21


,Program,Year,Period,Week Ending,Geo Level,State,State ANSI,Ag District,Ag District Code,County,...,Zip Code,Region,watershed_code,Watershed,Commodity,Data Item,Domain,Domain Category,Value,CV (%)
0,SURVEY,2024,YEAR,NaN,COUNTY,IOWA,19,NaN,99,OTHER COUNTIES,...,NaN,NaN,0,NaN,CORN,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",TOTAL,NOT SPECIFIED,205.7,0.5
1,SURVEY,2024,YEAR,NaN,COUNTY,IOWA,19,CENTRAL,50,BOONE,...,NaN,NaN,0,NaN,CORN,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",TOTAL,NOT SPECIFIED,217.6,2.3


In [ ]:
INPUT_FILE = f"{RAW_DIR}/nass_iowa_corn_yield_2019_2024.csv"
df = pd.read_csv(INPUT_FILE)

print("Original shape:", df.shape)
print("\nOriginal columns:")
print(df.columns.tolist())

display(df.head())



# We want:
# - Survey data
# - County level
# - Iowa
# - Corn
# - Annual
# - Grain yield
# - Bushels/acre

df_clean = df[
    (df["Program"] == "SURVEY") &
    (df["Geo Level"] == "COUNTY") &
    (df["State"] == "IOWA") &
    (df["Commodity"] == "CORN") &
    (df["Period"] == "YEAR") &
    (df["Data Item"].str.contains(
        "CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",
        case=False,
        na=False
    ))
].copy()

print("\nAfter basic filtering:", df_clean.shape)


# ------------------------------------------------
# Remove aggregate "OTHER COUNTIES" records
# ------------------------------------------------

df_clean = df_clean[
    df_clean["County ANSI"].notna() &
    (df_clean["County"].str.upper() != "OTHER COUNTIES")
].copy()

print("After removing aggregate rows:", df_clean.shape)


# ------------------------------------------------
# Clean year
# ------------------------------------------------

df_clean["Year"] = pd.to_numeric(
    df_clean["Year"],
    errors="coerce"
)

# Keep only our study period
df_clean = df_clean[
    df_clean["Year"].between(2019, 2024)
].copy()


# ------------------------------------------------
# Clean FIPS components
# ------------------------------------------------

# State ANSI = 19 for Iowa
df_clean["State ANSI"] = (
    pd.to_numeric(df_clean["State ANSI"], errors="coerce")
    .astype("Int64")
    .astype(str)
    .str.zfill(2)
)

# County ANSI may appear as 15.0, 1.0, etc.
df_clean["County ANSI"] = (
    pd.to_numeric(df_clean["County ANSI"], errors="coerce")
    .astype("Int64")
    .astype(str)
    .str.zfill(3)
)

# Create 5-digit county FIPS
df_clean["county_fips"] = (
    df_clean["State ANSI"] + df_clean["County ANSI"]
)


# ------------------------------------------------
# Clean yield value
# ------------------------------------------------

df_clean["yield_bu_ac"] = pd.to_numeric(
    df_clean["Value"],
    errors="coerce"
)


# ------------------------------------------------
# Select our final variables
# ------------------------------------------------

yield_clean = df_clean[
    [
        "county_fips",
        "State",
        "County",
        "Year",
        "yield_bu_ac"
    ]
].copy()

# Rename for consistency
yield_clean = yield_clean.rename(
    columns={
        "State": "state",
        "County": "county",
        "Year": "year"
    }
)

# Sort
yield_clean = yield_clean.sort_values(
    ["county_fips", "year"]
).reset_index(drop=True)


# ------------------------------------------------
# 9. Remove rows with missing target values
# ------------------------------------------------

missing_yield = yield_clean["yield_bu_ac"].isna().sum()

print("\nMissing yield values:", missing_yield)

yield_clean = yield_clean.dropna(
    subset=["yield_bu_ac"]
).copy()


# ------------------------------------------------
# Data-quality checks
# ------------------------------------------------

print("\n================ DATA QUALITY CHECKS ================")

print("\nYears present:")
print(sorted(yield_clean["year"].unique()))

print("\nNumber of unique counties:")
print(yield_clean["county_fips"].nunique())

print("\nDuplicate county-year rows:")
print(
    yield_clean.duplicated(
        subset=["county_fips", "year"]
    ).sum()
)

print("\nMissing values:")
print(yield_clean.isna().sum())

print("\nYield summary:")
print(yield_clean["yield_bu_ac"].describe())

print("\nFinal shape:", yield_clean.shape)


# ------------------------------------------------
# 11. Display cleaned data
# ------------------------------------------------

display(yield_clean.head(20))


# ------------------------------------------------
# 12. Save cleaned dataset
# ------------------------------------------------

OUTPUT_FILE = (
    f"{PROCESSED_DIR}/"
    "nass_iowa_corn_yield_2019_2024_clean.csv"
)

yield_clean.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nClean dataset saved to:")
print(OUTPUT_FILE)

Original shape: (533, 21)

Original columns:
['Program', 'Year', 'Period', 'Week Ending', 'Geo Level', 'State', 'State ANSI', 'Ag District', 'Ag District Code', 'County', 'County ANSI', 'Zip Code', 'Region', 'watershed_code', 'Watershed', 'Commodity', 'Data Item', 'Domain', 'Domain Category', 'Value', 'CV (%)']


,Program,Year,Period,Week Ending,Geo Level,State,State ANSI,Ag District,Ag District Code,County,...,Zip Code,Region,watershed_code,Watershed,Commodity,Data Item,Domain,Domain Category,Value,CV (%)
0,SURVEY,2024,YEAR,NaN,COUNTY,IOWA,19,NaN,99,OTHER COUNTIES,...,NaN,NaN,0,NaN,CORN,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",TOTAL,NOT SPECIFIED,205.7,0.5
1,SURVEY,2024,YEAR,NaN,COUNTY,IOWA,19,CENTRAL,50,BOONE,...,NaN,NaN,0,NaN,CORN,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",TOTAL,NOT SPECIFIED,217.6,2.3
2,SURVEY,2024,YEAR,NaN,COUNTY,IOWA,19,CENTRAL,50,HAMILTON,...,NaN,NaN,0,NaN,CORN,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",TOTAL,NOT SPECIFIED,212.3,1.6
3,SURVEY,2024,YEAR,NaN,COUNTY,IOWA,19,CENTRAL,50,MARSHALL,...,NaN,NaN,0,NaN,CORN,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",TOTAL,NOT SPECIFIED,222.2,2.6
4,SURVEY,2024,YEAR,NaN,COUNTY,IOWA,19,CENTRAL,50,POLK,...,NaN,NaN,0,NaN,CORN,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",TOTAL,NOT SPECIFIED,218.0,3.4



After basic filtering: (533, 21)
After removing aggregate rows: (524, 21)

Missing yield values: 0

================ DATA QUALITY CHECKS ================

Years present:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Number of unique counties:
99

Duplicate county-year rows:
0

Missing values:
county_fips    0
state          0
county         0
year           0
yield_bu_ac    0
dtype: int64

Yield summary:
count    524.000000
mean     195.406679
std       20.132931
min      132.700000
25%      182.500000
50%      198.750000
75%      210.100000
max      240.900000
Name: yield_bu_ac, dtype: float64

Final shape: (524, 5)


,county_fips,state,county,year,yield_bu_ac
0,19001,IOWA,ADAIR,2019,176.5
1,19001,IOWA,ADAIR,2020,168.7
2,19001,IOWA,ADAIR,2022,164.2
3,19001,IOWA,ADAIR,2024,211.2
4,19003,IOWA,ADAMS,2020,173.3
5,19005,IOWA,ALLAMAKEE,2019,187.2
6,19005,IOWA,ALLAMAKEE,2020,196.3
7,19005,IOWA,ALLAMAKEE,2021,198.5
8,19005,IOWA,ALLAMAKEE,2022,215.9
9,19005,IOWA,ALLAMAKEE,2023,183.9



Clean dataset saved to:
/content/drive/MyDrive/Project_2_Crop_Yield/data_processed/nass_iowa_corn_yield_2019_2024_clean.csv


In [ ]:


# 1. Missing values (NaN) in the existing data
print("Missing values in each column:")
print(yield_clean.isnull().sum())

print("\nTotal missing values in the entire dataset:")
print(yield_clean.isnull().sum().sum())


# 2. Missing county-year combinations
YEARS = list(range(2019, 2025))

all_counties = sorted(yield_clean["county_fips"].unique())

expected = pd.MultiIndex.from_product(
    [all_counties, YEARS],
    names=["county_fips", "year"]
).to_frame(index=False)

actual = yield_clean[["county_fips", "year"]].drop_duplicates()

missing_pairs = expected.merge(
    actual,
    on=["county_fips", "year"],
    how="left",
    indicator=True
)

missing_pairs = missing_pairs[
    missing_pairs["_merge"] == "left_only"
].drop(columns="_merge")


# 3. Summary
print("\n----------------------------------------")
print("COUNTY-YEAR COVERAGE")
print("----------------------------------------")

print("Unique counties:", len(all_counties))
print("Years expected:", YEARS)
print("Expected county-year combinations:", len(expected))
print("Actual county-year combinations:", len(actual))
print("Missing county-year combinations:", len(missing_pairs))


# 4. Show missing county-year combinations
print("\nMissing county-year combinations:")
display(
    missing_pairs.sort_values(
        ["county_fips", "year"]
    ).reset_index(drop=True)
)

Missing values in each column:
county_fips    0
state          0
county         0
year           0
yield_bu_ac    0
dtype: int64

Total missing values in the entire dataset:
0

----------------------------------------
COUNTY-YEAR COVERAGE
----------------------------------------
Unique counties: 99
Years expected: [2019, 2020, 2021, 2022, 2023, 2024]
Expected county-year combinations: 594
Actual county-year combinations: 524
Missing county-year combinations: 70

Missing county-year combinations:


,county_fips,year
0,19001,2021
1,19001,2023
2,19003,2019
3,19003,2021
4,19003,2022
...,...,...
65,19181,2024
66,19183,2021
67,19183,2024
68,19185,2023


In [ ]:
county_year_counts = (
    yield_clean.groupby("county_fips")["year"]
    .nunique()
    .value_counts()
    .sort_index()
)

print("Number of counties by number of available years:")
print(county_year_counts)

Number of counties by number of available years:
year
1     1
2     1
3     4
4     8
5    33
6    52
Name: count, dtype: int64


In [ ]:
# ============================================
# INSPECT INCOMPLETE COUNTIES
# ============================================

# Number of available years for each county
county_summary = (
    yield_clean
    .groupby(["county_fips", "county"])["year"]
    .agg(
        available_years="nunique",
        years_present=lambda x: sorted(x.tolist())
    )
    .reset_index()
)

# Keep only incomplete counties
incomplete = county_summary[
    county_summary["available_years"] < 6
].copy()

# Determine which years are missing
incomplete["missing_years"] = incomplete["years_present"].apply(
    lambda x: [year for year in YEARS if year not in x]
)

# Sort from least complete to most complete
incomplete = incomplete.sort_values(
    ["available_years", "county_fips"]
)

print("Incomplete counties:", len(incomplete))
print()

display(incomplete)

Incomplete counties: 47



,county_fips,county,available_years,years_present,missing_years
1,19003,ADAMS,1,[2020],"[2019, 2021, 2022, 2023, 2024]"
19,19039,CLARKE,2,"[2020, 2024]","[2019, 2021, 2022, 2023]"
31,19063,EMMET,3,"[2020, 2021, 2022]","[2019, 2023, 2024]"
42,19085,HARRISON,3,"[2020, 2022, 2023]","[2019, 2021, 2024]"
79,19159,RINGGOLD,3,"[2020, 2021, 2022]","[2019, 2023, 2024]"
90,19181,WARREN,3,"[2020, 2022, 2023]","[2019, 2021, 2024]"
0,19001,ADAIR,4,"[2019, 2020, 2022, 2024]","[2021, 2023]"
26,19053,DECATUR,4,"[2020, 2022, 2023, 2024]","[2019, 2021]"
38,19077,GUTHRIE,4,"[2019, 2020, 2021, 2022]","[2023, 2024]"
49,19099,JASPER,4,"[2019, 2021, 2022, 2023]","[2020, 2024]"


In [ ]:


# Make FIPS a string with 5 digits
yield_clean["county_fips"] = (
    pd.to_numeric(yield_clean["county_fips"], errors="coerce")
    .astype("Int64")
    .astype(str)
    .str.zfill(5)
)

# Make sure year is integer
yield_clean["year"] = (
    pd.to_numeric(yield_clean["year"], errors="coerce")
    .astype(int)
)

# Sort the dataset
yield_clean = yield_clean.sort_values(
    ["county_fips", "year"]
).reset_index(drop=True)

# Final column order
yield_clean = yield_clean[
    [
        "county_fips",
        "state",
        "county",
        "year",
        "yield_bu_ac"
    ]
]

# Final checks
print("Final shape:", yield_clean.shape)
print("Unique counties:", yield_clean["county_fips"].nunique())
print("Years:", sorted(yield_clean["year"].unique()))
print("Missing values:", yield_clean.isna().sum().sum())
print(
    "Duplicate county-year rows:",
    yield_clean.duplicated(
        ["county_fips", "year"]
    ).sum()
)

display(yield_clean.head(10))

# Save final target table
FINAL_FILE = (
    f"{PROCESSED_DIR}/"
    "nass_iowa_corn_yield_2019_2024_clean.csv"
)

yield_clean.to_csv(FINAL_FILE, index=False)

print("\nSaved to:")
print(FINAL_FILE)

Final shape: (524, 5)
Unique counties: 99
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Missing values: 0
Duplicate county-year rows: 0


,county_fips,state,county,year,yield_bu_ac
0,19001,IOWA,ADAIR,2019,176.5
1,19001,IOWA,ADAIR,2020,168.7
2,19001,IOWA,ADAIR,2022,164.2
3,19001,IOWA,ADAIR,2024,211.2
4,19003,IOWA,ADAMS,2020,173.3
5,19005,IOWA,ALLAMAKEE,2019,187.2
6,19005,IOWA,ALLAMAKEE,2020,196.3
7,19005,IOWA,ALLAMAKEE,2021,198.5
8,19005,IOWA,ALLAMAKEE,2022,215.9
9,19005,IOWA,ALLAMAKEE,2023,183.9



Saved to:
/content/drive/MyDrive/Project_2_Crop_Yield/data_processed/nass_iowa_corn_yield_2019_2024_clean.csv
